# 📚 Section 7: Vectorization & Broadcasting
> Vectorization & Broadcasting — The two reasons NumPy runs 50-200x faster than a Python loop

---
# 🎯 Learning Objective
Today I want to learn:
- [x] Why vectorized operations (`arr * 1.1`, `np.sqrt(arr)`, etc.) replace explicit loops and run dramatically faster
- [x] The 3-step Broadcasting compatibility rule for operating on arrays of different shapes
- [x] How to apply row-wise vs column-wise Broadcasting using `reshape(-1, 1)` for normalization and achievement-rate patterns

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

Vectorization means applying one operation to an entire array at once, with no explicit loop — `data * 1.1` instead of looping and multiplying each element. Broadcasting is what makes this work even when the two arrays involved have different shapes: NumPy automatically (and virtually, without actually copying data) stretches the smaller array to match the larger one, following a simple rule.

Vectorization(벡터화)은 명시적인 반복문 없이 배열 전체에 한 번에 연산을 적용하는 것입니다 — 각 원소를 반복문으로 곱하는 대신 `data * 1.1` 한 줄로 처리합니다. Broadcasting(브로드캐스팅)은 두 배열의 shape이 달라도 이것이 가능하게 해주는 원리로, NumPy가 (실제로 데이터를 복사하지 않고 가상으로) 작은 배열을 큰 배열에 맞춰 자동으로 확장하며, 이는 간단한 규칙을 따릅니다.

### Two reasons NumPy is fast / NumPy가 빠른 두 가지 이유

| Concept / 개념 | Solves / 해결하는 문제 | Example / 예시 |
|---|---|---|
| Vectorization | Apply ONE operation to every element, no loop (반복문 없이 모든 원소에 연산 적용) | `data * 1.1` |
| Broadcasting | Operate between arrays of DIFFERENT shapes (다른 shape의 배열 간 연산) | `(5,3) - (3,)` |

### Broadcasting compatibility rule (compare dimensions from the right) / 브로드캐스팅 호환 규칙 (오른쪽 차원부터 비교)
1. Equal size in that dimension → operate as-is (크기가 같으면 → 그대로 연산)
2. One side is `1` → it stretches to match the other (한쪽이 1이면 → 상대방 크기로 확장)
3. Neither equal nor `1` → `ValueError` (둘 다 아니면 → 에러)

## Why do we use it?
*(When is it useful?)*

Because together they're the reason a NumPy one-liner beats a Python loop by 50-200x, and because most real preprocessing — normalizing a column, subtracting a per-row target, applying a per-column weight — is naturally a "different-shaped arrays" problem that Broadcasting solves in one line.

이 둘을 합치면 NumPy 한 줄이 Python 반복문보다 50~200배 빠른 이유가 되며, 실제 전처리 작업 대부분 — 컬럼 정규화, 행별 목표값 빼기, 열별 가중치 적용 — 이 본질적으로 "서로 다른 shape의 배열" 문제이고 Broadcasting이 이를 한 줄로 해결하기 때문입니다.

## When is it used in Business Analytics?
*(Real-world use case)*

- Applying VAT, a discount, or any formula to an entire column of a million rows in one line.  
  백만 행짜리 컬럼 전체에 부가세·할인 등 공식을 한 줄로 적용할 때.
- Min-Max or Z-score normalizing every column of a dataset before modeling.  
  모델링 전 데이터셋의 모든 컬럼을 Min-Max나 Z-score로 정규화할 때.
- Computing each row's achievement rate against a DIFFERENT target per row.  
  행마다 다른 목표값 대비 달성률을 계산할 때.
- Calculating a KPI (RPV, ROAS, LTV) for every customer at once instead of looping through them.  
  고객 한 명씩 반복하지 않고 전체 고객의 KPI(RPV, ROAS, LTV)를 한 번에 계산할 때.

---
# 📝 Syntax

## Basic Syntax

In [1]:
import numpy as np

data = np.array([1200, 1350, 1100, 1400])

# Vectorization — one operation, whole array, no loop
print(data * 1.1)          # VAT applied to everything at once
print(np.sqrt(data))        # math functions work the same way

# Broadcasting — scalar "stretches" to match the array
print(data + 100)

# Broadcasting — a smaller array stretches to match a bigger one
matrix = np.array([[1, 2, 3], [4, 5, 6]])
row = np.array([10, 20, 30])         # shape (3,) matches matrix's columns
print(matrix + row)

[1320. 1485. 1210. 1540.]
[34.64101615 36.74234614 33.1662479  37.41657387]
[1300 1450 1200 1500]
[[11 22 33]
 [14 25 36]]


## Common Variations

In [2]:
import numpy as np

matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

# row-wise broadcasting (shape (3,) applies across columns)
row_vals = np.array([10, 20, 30])
print(matrix + row_vals)

# column-wise broadcasting needs reshape(-1, 1) first
col_vals = np.array([100, 200, 300]).reshape(-1, 1)   # shape (3, 1)
print(matrix + col_vals)

# incompatible shapes raise ValueError
try:
    np.zeros((3, 4)) + np.array([1, 2, 3])
except ValueError as e:
    print("ValueError:", e)

# vectorized comparison returns a boolean array, ready for filtering
print(matrix > 5)

[[11 22 33]
 [14 25 36]
 [17 28 39]]
[[101 102 103]
 [204 205 206]
 [307 308 309]]
ValueError: operands could not be broadcast together with shapes (3,4) (3,) 
[[False False False]
 [False False  True]
 [ True  True  True]]


---
# 🧪 Small Examples

## Example 1: Vectorization — 10,000 Customers' KPIs, No Loops
### 7-1. Vectorization — 루프 없이 전체 배열 연산

In [3]:
import numpy as np

# Business: compute KPIs for 10,000 customers at once
np.random.seed(42)
n_customers = 10000

revenue = np.random.randint(10000, 500000, n_customers)
visits = np.random.randint(1, 30, n_customers)
ad_spend = np.random.randint(5000, 50000, n_customers)

# Vectorized KPI calculations
rpv = revenue / visits                     # Revenue Per Visit
roas = revenue / ad_spend                   # Return on Ad Spend
log_revenue = np.log1p(revenue)             # log(1+x), safe for values >= 0
above_avg = revenue > revenue.mean()        # vectorized boolean comparison

print(f"Customers: {n_customers:,}")
print(f"Average revenue: {revenue.mean():,.0f}")
print(f"Average ROAS: {roas.mean():.2f}x")
print(f"Above-average customers: {above_avg.mean() * 100:.1f}%")
print(f"Highest RPV: {rpv.max():,.0f}/visit")

Customers: 10,000
Average revenue: 255,834
Average ROAS: 13.10x
Above-average customers: 50.0%
Highest RPV: 499,197/visit


## Example 2: Broadcasting — Min-Max Normalization & Per-Row Achievement Rate
### 7-2. Broadcasting — 다른 shape 배열 간 자동 연산

In [4]:
import numpy as np

# Business pattern 1: Min-Max normalization (0-1 scaling) per column
data = np.array([
    [1200, 3200, 3.8],
    [1350, 3500, 3.9],
    [1100, 2800, 3.9],
    [1400, 3700, 3.8],
    [1600, 4100, 4.0],
])   # shape (5, 3)

col_min = data.min(axis=0)   # shape (3,)
col_max = data.max(axis=0)   # shape (3,)

normalized = (data - col_min) / (col_max - col_min)   # Broadcasting: (5,3) with (3,)
print("Normalized:\n", np.round(normalized, 3))

# Business pattern 2: per-row achievement rate against a DIFFERENT target per row
actual = np.array([
    [1200, 800, 600],
    [1400, 950, 720],
    [1300, 870, 680],
    [1700, 1100, 900],
])   # shape (4, 3): 4 quarters x 3 regions

targets = np.array([1500, 1600, 1400, 1800]).reshape(-1, 1)   # shape (4, 1)

achievement = actual / targets * 100   # Broadcasting: (4,3) with (4,1)
print("\nAchievement rate (%):\n", np.round(achievement, 1))

Normalized:
 [[0.2   0.308 0.   ]
 [0.5   0.538 0.5  ]
 [0.    0.    0.5  ]
 [0.6   0.692 0.   ]
 [1.    1.    1.   ]]

Achievement rate (%):
 [[80.  53.3 40. ]
 [87.5 59.4 45. ]
 [92.9 62.1 48.6]
 [94.4 61.1 50. ]]


## Example 3: Practice — One Mini-Exercise per Subtopic
### 연습 문제 (Practice Problems, 7-1 ~ 7-2)

**Task / 과제:** This section's original material has one practice problem per subtopic — solve both below.
이번 섹션은 하위주제마다 연습 문제가 하나씩 있습니다 — 아래 2개를 모두 풀어보세요.

1. **(7-1)** From 10,000 random sales values (10,000–1,000,000), compute WITHOUT a loop: (a) +10% VAT, (b) the total sum, (c) the count of values above the mean.
   1만 개의 랜덤 매출 데이터(10,000~1,000,000)에서 반복문 없이 (a) 부가세 10% 추가, (b) 전체 합계, (c) 평균보다 큰 값의 개수를 계산하세요.
2. **(7-2)** From a 5-product × 4-month sales table, divide each row by its own max (row-wise Broadcasting) to get relative share (0–1); each row's max should become `1.0`.
   5개 상품×4개월 판매량 배열에서 각 행을 그 행의 최댓값으로 나누어(행 방향 Broadcasting) 상대 판매 비중(0~1)을 계산하세요. 각 행의 최댓값은 `1.0`이 되어야 합니다.

Fill in each `________` below, then run the cell.
아래 각 `________`를 채운 후 셀을 실행하세요.

In [ ]:
# ✏️ Practice — replace each ________ line below, then run this cell.
# ✏️ 연습 문제 — 아래 각 ________ 줄을 채운 후 셀을 실행하세요.

import numpy as np

print("--- 7-1: Vectorization ---")
# TODO: np.random.seed(0); sales = np.random.randint(10000, 1000001, size=10000)
# then print: sales*1.1 (first 5), np.sum(sales), count where sales > sales.mean()
np.random.seed(0)
sales = np.random.randint(10000, 1000001, size=10000)
print(sales*1.1)
print(np.sum(sales))
print(np.sum(sales > sales.mean()))

print("\n--- 7-2: Broadcasting ---")
# TODO: 5x4 product sales table -> divide each row by its own max (reshape(-1,1) first), print result + row-max check
sales = np.array([
    [120, 200, 150, 180],
    [300, 280, 350, 400],
    [500, 450, 470, 520],
    [90, 100, 80, 110],
    [700, 650, 720, 680],
])
print(sales / sales.max(axis=1, keepdims=True))
print(sales / sales.max(axis=1, keepdims=True) == 1.0)

print("\n✅ Fill in each ________ above with real code, then re-run to see both results.")
print("✅ 위 각 ________ 를 실제 코드로 채운 뒤 다시 실행하면 두 결과를 모두 볼 수 있습니다.")

--- 7-1: Vectorization ---

--- 7-2: Broadcasting ---

✅ Fill in each ________ above with real code, then re-run to see both results.
✅ 위 각 ________ 를 실제 코드로 채운 뒤 다시 실행하면 두 결과를 모두 볼 수 있습니다.


---
# ⚠️ Common Mistakes

**Mistake 1 — Reaching for a loop out of habit.**
Even on small arrays, it's easy to default to a `for` loop just because that's the general Python instinct — missing an easy vectorized one-liner.  
작은 배열에서도 일반적인 파이썬 습관 때문에 `for` 문을 먼저 떠올리기 쉽습니다 — 간단한 벡터화 한 줄로 될 일을 놓치게 됩니다.  
✅ **Fix:** Before writing a loop over a NumPy array, pause and ask whether the operation can be written directly on the whole array instead.  
✅ **해결법:** NumPy 배열에 반복문을 쓰기 전, 그 연산을 배열 전체에 바로 적용할 수는 없는지 먼저 생각해보세요.

**Mistake 2 — Calling `np.log()` on data that includes 0.**
`np.log(0)` returns `-inf` (with a runtime warning), not an error — which can silently poison downstream sums or means if you don't notice it.  
`np.log(0)`은 에러가 아니라 (런타임 경고와 함께) `-inf`를 반환합니다 — 눈치채지 못하면 이후의 합계나 평균 계산이 조용히 오염될 수 있습니다.  
✅ **Fix:** For data that may include 0, use `np.log1p(x)` (computes `log(1 + x)` safely) instead of `np.log(x)`.  
✅ **해결법:** 0이 포함될 수 있는 데이터에는 `np.log(x)` 대신 `np.log1p(x)`(`log(1 + x)`를 안전하게 계산)를 사용하세요.

**Mistake 3 — Being surprised that dividing integer arrays gives floats.**
`np.array([10, 20]) / np.array([2, 4])` returns `[5. 5.]` as `float64` — NumPy's `/` always does true division, even between two integer arrays.  
`np.array([10, 20]) / np.array([2, 4])`는 `float64` 타입의 `[5. 5.]`를 반환합니다 — NumPy의 `/`는 두 정수 배열 사이에서도 항상 실수 나눗셈을 수행합니다.  
✅ **Fix:** If you specifically need integer division, use `//` instead of `/`.  
✅ **해결법:** 정수 나눗셈이 꼭 필요하다면 `/` 대신 `//`를 사용하세요.

**Mistake 4 — Confusing row-wise and column-wise Broadcasting.**
A 1D array of shape `(3,)` broadcasts across COLUMNS (applies the same values to every row); to apply different values to every ROW instead, you need a column vector — `reshape(-1, 1)` — not a plain 1D array.  
shape이 `(3,)`인 1D 배열은 열 방향으로 브로드캐스트됩니다(모든 행에 동일한 값 적용) — 반대로 행마다 다른 값을 적용하려면 평범한 1D 배열이 아니라 `reshape(-1, 1)`로 만든 열벡터가 필요합니다.  
✅ **Fix:** Row-wise application (same per column) → keep it 1D. Column-wise application (same per row) → reshape to `(-1, 1)` first.  
✅ **해결법:** 열 기준 적용(각 열에 동일)은 1D 그대로, 행 기준 적용(각 행에 동일)은 먼저 `(-1, 1)`로 reshape하세요.

**Mistake 5 — Misreading a Broadcasting `ValueError`.**
An error like `shapes (3,4) (3,)` lists the ORIGINAL shapes involved — to understand why they failed, compare them from the RIGHTMOST dimension, per the 3-step rule.  
`shapes (3,4) (3,)` 같은 에러 메시지는 관련된 원본 shape을 그대로 나열한 것입니다 — 왜 실패했는지 이해하려면 3단계 규칙에 따라 가장 오른쪽 차원부터 비교하세요.  
✅ **Fix:** Write out both shapes and align them from the right before deciding what needs to change.  
✅ **해결법:** 무엇을 바꿔야 할지 판단하기 전에, 두 shape을 오른쪽부터 나란히 정렬해서 적어보세요.

**Mistake 6 — Assuming Broadcasting physically duplicates data in memory.**
Broadcasting doesn't actually copy the smaller array to match the larger one's size — the expansion is virtual, computed on the fly, which is part of why it's fast and memory-efficient.  
Broadcasting은 작은 배열을 실제로 복사해서 큰 배열 크기로 만드는 게 아닙니다 — 확장은 가상으로, 연산 중에 즉석에서 이루어지며, 이것이 빠르고 메모리 효율적인 이유 중 하나입니다.  
✅ **Fix:** No action needed — this is a reason to trust Broadcasting for large data, not a trap to avoid.  
✅ **해결법:** 딱히 조치가 필요하진 않습니다 — 이건 피해야 할 함정이 아니라, 대용량 데이터에도 Broadcasting을 믿고 써도 되는 이유입니다.

---
# 💡 Tips
Useful tips or shortcuts

- Before writing any loop over a NumPy array, ask "is there a vectorized way to write this?" — the answer is almost always yes.  
  NumPy 배열에 반복문을 쓰기 전 "벡터화된 방법이 있지 않을까?"라고 먼저 물어보세요 — 대부분의 경우 답은 "있다"입니다.
- When a Broadcasting operation doesn't behave as expected, print both arrays' `.shape` side by side before debugging further.  
  Broadcasting 연산이 예상대로 동작하지 않을 때는, 더 디버깅하기 전에 두 배열의 `.shape`을 나란히 출력해보세요.
- `reshape(-1, 1)` is the single most useful trick for turning a per-row 1D array (like targets or maxes) into something Broadcasting can apply row-by-row.  
  `reshape(-1, 1)`은 행별 1D 배열(목표값, 최댓값 등)을 Broadcasting이 행 단위로 적용할 수 있게 만드는 가장 유용한 트릭입니다.

---
# 🔗 Related Concepts

```
Section 6 — Array Manipulation
   (reshape, flatten/ravel, transpose, concatenate, split)
        ↓
🔵 Section 7 — Vectorization & Broadcasting   ← you are here
   (loop-free operations, automatic shape matching)
        ↓
Section 8 — Aggregation
   (sum, mean, std, min/max, percentile — often paired with Broadcasting)
        ↓
Section 9 — Random / Sorting / Filtering
        ↓
Section 10 — Pandas + NumPy integration
        ↓
Sections 11-12 — Business KPIs, missing values
        ↓
Pandas → SQL → Tableau
```

*How is today's topic connected to other concepts?*

Section 6's `reshape(-1, 1)` was the setup; this section is the payoff — that column vector exists specifically to make row-wise Broadcasting work. Going forward, Section 8's aggregation functions (`.mean(axis=0)`, `.max(axis=1)`, etc.) are almost always the SOURCE of the small array that then gets broadcast back against the original data, exactly like `col_min`/`col_max`/`targets` did in this section's examples.

섹션 6의 `reshape(-1, 1)`이 준비 과정이었다면, 이번 섹션은 그 결과물입니다 — 그 열벡터는 바로 행 방향 Broadcasting을 위해 존재합니다. 앞으로 섹션 8의 집계 함수(`.mean(axis=0)`, `.max(axis=1)` 등)는 거의 항상 이번 섹션의 `col_min`/`col_max`/`targets`처럼, 원본 데이터에 다시 브로드캐스트되어 쓰이는 작은 배열의 출처가 됩니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
You're preparing a dataset for a churn-prediction model and a quarterly performance report. You need to (1) compute a KPI for every customer without a loop, (2) Min-Max normalize every feature column so the model isn't biased by scale, and (3) compute each quarter's achievement rate against that quarter's own (different) target.  
이탈 예측 모델용 데이터셋과 분기 실적 리포트를 준비하는 상황입니다. 필요한 것: (1) 반복문 없이 전체 고객의 KPI 계산, (2) 모델이 스케일에 편향되지 않도록 모든 특성 컬럼을 Min-Max 정규화, (3) 분기마다 다른 목표값 대비 달성률 계산.  

**To-do / 할 일:**
- [x] Vectorize a per-customer KPI calculation across a large customer array.  
      대량 고객 배열 전체에 고객별 KPI 계산을 벡터화합니다.
- [x] Min-Max normalize every column of a feature matrix using Broadcasting.  
      Broadcasting으로 특성 행렬의 모든 컬럼을 Min-Max 정규화합니다.
- [x] Divide each quarter's actuals by that quarter's own target (column vector Broadcasting).   
      각 분기의 실적을 그 분기의 목표값으로 나눕니다(열벡터 Broadcasting).

In [7]:
import numpy as np

# 1. Vectorized KPI across customers
np.random.seed(3)
revenue = np.random.randint(20000, 400000, size=8)
visits = np.random.randint(2, 20, size=8)
rpv = revenue / visits
print("Revenue per visit:", np.round(rpv, 0))

# 2. Min-Max normalize every feature column
features = np.array([
    [1200, 3200, 3.8],
    [900, 2600, 3.5],
    [1500, 3900, 4.1],
    [1100, 3000, 3.7],
])
col_min, col_max = features.min(axis=0), features.max(axis=0)
normalized = (features - col_min) / (col_max - col_min)
print("\nNormalized features:\n", np.round(normalized, 3))

# 3. Per-quarter achievement rate against a different target each quarter
actual = np.array([[1200, 800], [1400, 950], [1300, 870], [1700, 1100]])
targets = np.array([1500, 1600, 1400, 1800]).reshape(-1, 1)
achievement = actual / targets * 100
print("\nAchievement rate (%):\n", np.round(achievement, 1))

Revenue per visit: [  7041.  19845.   8087.   8507. 193424.   2233.  26189.  15439.]

Normalized features:
 [[0.5   0.462 0.5  ]
 [0.    0.    0.   ]
 [1.    1.    1.   ]
 [0.333 0.308 0.333]]

Achievement rate (%):
 [[80.  53.3]
 [87.5 59.4]
 [92.9 62.1]
 [94.4 61.1]]


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

Vectorization applies one operation to an entire array at once — no explicit loop — which is the core reason NumPy runs 50-200x faster than equivalent Python code. Broadcasting is what lets Vectorization work between arrays of DIFFERENT shapes: NumPy virtually (not physically) stretches the smaller array to match the larger one, following a 3-step rule compared from the rightmost dimension — equal sizes operate directly, a size of `1` stretches to match, and anything else raises a `ValueError`. The single most useful pattern that falls out of this is `reshape(-1, 1)` to turn a per-row array (a target, a max, a mean) into something that broadcasts correctly against every row of a 2D array — the foundation of Min-Max normalization, Z-score standardization, and per-row achievement rates.

Vectorization은 반복문 없이 배열 전체에 한 번에 연산을 적용하며, 이것이 NumPy가 동등한 파이썬 코드보다 50~200배 빠른 핵심 이유입니다. Broadcasting은 서로 다른 shape의 배열 사이에서도 Vectorization이 동작하게 해주는 원리로, NumPy가 작은 배열을 (실제가 아니라) 가상으로 큰 배열에 맞춰 확장하며, 오른쪽 차원부터 비교하는 3단계 규칙을 따릅니다 — 크기가 같으면 그대로 연산, 크기가 1이면 확장, 그 외에는 `ValueError`. 여기서 나오는 가장 유용한 패턴 하나는 `reshape(-1, 1)`로, 행별 배열(목표값, 최댓값, 평균)을 2D 배열의 모든 행에 올바르게 브로드캐스트되도록 만들어줍니다 — Min-Max 정규화, Z-score 표준화, 행별 달성률 계산의 기초입니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence.

> Vectorization drops the loop, and Broadcasting is the rule that lets differently-shaped arrays still operate together — together they're why NumPy is dramatically faster than plain Python.

> Vectorization은 반복문을 없애고, Broadcasting은 서로 다른 shape의 배열도 함께 연산할 수 있게 해주는 규칙입니다 — 이 둘이 합쳐져 NumPy가 순수 파이썬보다 훨씬 빠른 이유가 됩니다.

---
# ❓ Review Questions

**Q1.** Why is `data * 1.1` faster than looping through `data` and multiplying each element by `1.1`?
`data * 1.1`이 `data`를 반복문으로 돌며 각 원소에 `1.1`을 곱하는 것보다 왜 더 빠른가요?

**Q2.** What are the 3 steps for checking whether two shapes are Broadcasting-compatible?
두 shape이 Broadcasting 호환이 되는지 확인하는 3단계는 무엇인가요?

**Q3.** Why does adding a `(3,)` array to a `(3, 3)` matrix apply to every ROW, while adding a `(3, 1)` array applies to every COLUMN?
`(3,)` 배열을 `(3, 3)` 행렬에 더하면 왜 모든 행에 적용되고, `(3, 1)` 배열을 더하면 왜 모든 열에 적용되나요?

**Q4.** What does `np.log(0)` actually return, and what should you use instead if your data might contain 0?
`np.log(0)`은 실제로 무엇을 반환하며, 데이터에 0이 포함될 수 있다면 대신 무엇을 사용해야 하나요?

**Q5.** In the Min-Max normalization pattern `(data - col_min) / (col_max - col_min)`, what shape are `col_min` and `col_max`, and why does that shape work with `data`?
Min-Max 정규화 패턴 `(data - col_min) / (col_max - col_min)`에서 `col_min`과 `col_max`의 shape은 무엇이며, 왜 그 shape이 `data`와 함께 동작하나요?

---
*📅 Try answering these again in a few days.*